## Importing Libraries and Data

In [29]:
import pandas as pd

# Load dataset
df = pd.read_csv('/kaggle/input/groceries/groceries - groceries.csv')
# df_strip = df.sample(60)
# Convert dataframe into a list of transactions
transactions = []
for index, row in df_strip.iterrows():
    basket = row.dropna().tolist()[1:]  # Exclude 'Item(s)' column
    transactions.append(basket)

# Display a sample transaction
print(transactions[:15])

[['meat', 'whole milk', 'domestic eggs', 'pastry'], ['sausage', 'pip fruit', 'whole milk', 'sliced cheese', 'frozen vegetables', 'frozen meals', 'napkins', 'shopping bags'], ['pastry', 'bottled beer', 'liquor', 'red/blush wine'], ['shopping bags'], ['bottled water', 'misc. beverages', 'cooking chocolate'], ['whole milk', 'ice cream', 'pastry', 'coffee', 'shopping bags'], ['frankfurter', 'white bread', 'margarine', 'shopping bags'], ['other vegetables', 'whole milk', 'cream cheese', 'frozen meals', 'rolls/buns', 'brown bread', 'bottled water', 'newspapers'], ['whole milk', 'long life bakery product', 'chocolate'], ['citrus fruit', 'berries', 'other vegetables', 'whole milk', 'frozen meals', 'newspapers'], ['tropical fruit', 'pasta', 'candy'], ['bottled beer', 'napkins'], ['frankfurter', 'other vegetables', 'yogurt', 'beverages', 'hard cheese', 'rolls/buns', 'canned fish', 'soda', 'canned beer', 'brandy', 'shopping bags'], ['rolls/buns', 'bottled beer'], ['napkins']]


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


## Implementation of Apriori

In [30]:
from mlxtend.frequent_patterns import apriori
from collections import deque
import pandas as pd

# Debug: Check first few transactions
print("Sample Transactions:", transactions[:5])

def process_transactions(transactions, window_size=100, min_support=0.001):
    window = deque(maxlen=window_size)
    freq_itemsets = []
    
    # Process transactions in streaming windows
    for i, basket in enumerate(transactions):
        window.append(basket)
        
        # When window is full, process it
        if len(window) == window_size:
            df_window = pd.DataFrame([{item: True for item in basket} for basket in window]).fillna(False)
            
            # Debug: Check one-hot encoded data
            print("\nOne-Hot Encoded Data (Full Window Sample):")
            print(df_window.head())
            
            frequent_itemsets = apriori(df_window, min_support=min_support, use_colnames=True)
            
            # Debug: Check if frequent itemsets are found
            if not frequent_itemsets.empty:
                print("\nFrequent Itemsets Found in Full Window:")
                print(frequent_itemsets.head())
            else:
                print("\nNo frequent itemsets found in this full window.")
            
            freq_itemsets.append(frequent_itemsets)
    
    # Process any remaining transactions (final partial window)
    if window and len(window) < window_size:
        df_window = pd.DataFrame([{item: True for item in basket} for basket in window]).fillna(False)
        print("\nProcessing Final Partial Window:")
        print(df_window.head())
        frequent_itemsets = apriori(df_window, min_support=min_support, use_colnames=True)
        if not frequent_itemsets.empty:
            print("\nFrequent Itemsets Found in Final Partial Window:")
            print(frequent_itemsets.head())
        else:
            print("\nNo frequent itemsets found in the final partial window.")
        freq_itemsets.append(frequent_itemsets)
    
    return freq_itemsets

# For testing with your sample transactions, use window_size equal to the number of transactions
streaming_results = process_transactions(transactions, window_size=len(transactions), min_support=0.001)

# Display the results from the processed window(s)
print("\nSample Frequent Itemsets (Streaming Apriori):")
for i, res in enumerate(streaming_results):
    print(f"\nWindow {i+1}:")
    print(res)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<ipython-input-30-f75a323bb00b>:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_window = pd.DataFrame([{item: True for item in basket} for basket in window]).fillna(False)


Sample Transactions: [['meat', 'whole milk', 'domestic eggs', 'pastry'], ['sausage', 'pip fruit', 'whole milk', 'sliced cheese', 'frozen vegetables', 'frozen meals', 'napkins', 'shopping bags'], ['pastry', 'bottled beer', 'liquor', 'red/blush wine'], ['shopping bags'], ['bottled water', 'misc. beverages', 'cooking chocolate']]

One-Hot Encoded Data (Full Window Sample):
    meat  whole milk  domestic eggs  pastry  sausage  pip fruit  \
0   True        True           True    True    False      False   
1  False        True          False   False     True       True   
2  False       False          False    True    False      False   
3  False       False          False   False    False      False   
4  False       False          False   False    False      False   

   sliced cheese  frozen vegetables  frozen meals  napkins  ...   beef  \
0          False              False         False    False  ...  False   
1           True               True          True     True  ...  False   
2 

### Run traditional apriori on full dataset

In [31]:
# Traditional Apriori on the full dataset
df_full = pd.DataFrame([{item: True for item in basket} for basket in transactions]).fillna(False)
traditional_frequent_itemsets = apriori(df_full, min_support=0.001, use_colnames=True)

print("Frequent Itemsets (Traditional Apriori):")
print(traditional_frequent_itemsets.head(10))

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<ipython-input-31-7a4ac927239f>:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_full = pd.DataFrame([{item: True for item in basket} for basket in transactions]).fillna(False)


Frequent Itemsets (Traditional Apriori):
    support             itemsets
0  0.033333               (meat)
1  0.233333         (whole milk)
2  0.050000      (domestic eggs)
3  0.133333             (pastry)
4  0.083333            (sausage)
5  0.100000          (pip fruit)
6  0.033333      (sliced cheese)
7  0.050000  (frozen vegetables)
8  0.066667       (frozen meals)
9  0.083333            (napkins)


### Aggregate Streaming Apriori

In [32]:
# Concatenate all streaming results into one DataFrame
streaming_all = pd.concat(streaming_results, ignore_index=True)

# Remove duplicate itemsets (this considers the itemset tuple)
streaming_unique = streaming_all.drop_duplicates(subset=['itemsets'])

print("\nFrequent Itemsets (Aggregated from Streaming Apriori):")
print(streaming_unique.head(10))


Frequent Itemsets (Aggregated from Streaming Apriori):
    support             itemsets
0  0.033333               (meat)
1  0.233333         (whole milk)
2  0.050000      (domestic eggs)
3  0.133333             (pastry)
4  0.083333            (sausage)
5  0.100000          (pip fruit)
6  0.033333      (sliced cheese)
7  0.050000  (frozen vegetables)
8  0.066667       (frozen meals)
9  0.083333            (napkins)


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


### Comparision Analysis

In [33]:
# Count of frequent itemsets
print("\nNumber of frequent itemsets (Traditional):", len(traditional_frequent_itemsets))
print("Number of unique frequent itemsets (Streaming):", len(streaming_unique))

# Create sets for comparison
trad_itemsets = set(traditional_frequent_itemsets['itemsets'].apply(lambda x: tuple(sorted(x))))
stream_itemsets = set(streaming_unique['itemsets'].apply(lambda x: tuple(sorted(x))))

# Common itemsets
common_itemsets = trad_itemsets.intersection(stream_itemsets)
print("\nNumber of common frequent itemsets:", len(common_itemsets))

# Unique to each approach
unique_to_traditional = trad_itemsets - stream_itemsets
unique_to_streaming = stream_itemsets - trad_itemsets

print("Itemsets unique to Traditional Apriori:", unique_to_traditional)
print("Itemsets unique to Streaming Apriori:", unique_to_streaming)


Number of frequent itemsets (Traditional): 30374
Number of unique frequent itemsets (Streaming): 30374

Number of common frequent itemsets: 30374
Itemsets unique to Traditional Apriori: set()
Itemsets unique to Streaming Apriori: set()


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [34]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from collections import deque

# Assume 'transactions' is already defined from previous parts

# --- Traditional Apriori ---
df_full = pd.DataFrame([{item: True for item in basket} for basket in transactions]).fillna(False)
traditional_frequent_itemsets = apriori(df_full, min_support=0.001, use_colnames=True)

print("Frequent Itemsets (Traditional Apriori):")
print(traditional_frequent_itemsets.head(10))


# --- Aggregated Streaming Apriori ---
# 'streaming_results' is the list of DataFrames from the streaming implementation.
streaming_all = pd.concat(streaming_results, ignore_index=True)
streaming_unique = streaming_all.drop_duplicates(subset=['itemsets'])

print("\nFrequent Itemsets (Aggregated from Streaming Apriori):")
print(streaming_unique.head(10))

# --- Comparison ---
print("\nNumber of frequent itemsets (Traditional):", len(traditional_frequent_itemsets))
print("Number of unique frequent itemsets (Streaming):", len(streaming_unique))

# Prepare sets for comparison
trad_itemsets = set(traditional_frequent_itemsets['itemsets'].apply(lambda x: tuple(sorted(x))))
stream_itemsets = set(streaming_unique['itemsets'].apply(lambda x: tuple(sorted(x))))

common_itemsets = trad_itemsets.intersection(stream_itemsets)
print("\nNumber of common frequent itemsets:", len(common_itemsets))

unique_to_traditional = trad_itemsets - stream_itemsets
unique_to_streaming = stream_itemsets - trad_itemsets

print("\nItemsets unique to Traditional Apriori:")
print(unique_to_traditional)

print("\nItemsets unique to Streaming Apriori:")
print(unique_to_streaming)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<ipython-input-34-974da6984560>:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_full = pd.DataFrame([{item: True for item in basket} for basket in transactions]).fillna(False)


Frequent Itemsets (Traditional Apriori):
    support             itemsets
0  0.033333               (meat)
1  0.233333         (whole milk)
2  0.050000      (domestic eggs)
3  0.133333             (pastry)
4  0.083333            (sausage)
5  0.100000          (pip fruit)
6  0.033333      (sliced cheese)
7  0.050000  (frozen vegetables)
8  0.066667       (frozen meals)
9  0.083333            (napkins)

Frequent Itemsets (Aggregated from Streaming Apriori):
    support             itemsets
0  0.033333               (meat)
1  0.233333         (whole milk)
2  0.050000      (domestic eggs)
3  0.133333             (pastry)
4  0.083333            (sausage)
5  0.100000          (pip fruit)
6  0.033333      (sliced cheese)
7  0.050000  (frozen vegetables)
8  0.066667       (frozen meals)
9  0.083333            (napkins)

Number of frequent itemsets (Traditional): 30374
Number of unique frequent itemsets (Streaming): 30374

Number of common frequent itemsets: 30374

Itemsets unique to Traditiona